In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from dotenv import load_dotenv
import os
from pathlib import Path
import sys

load_dotenv()

import os

sys.path.append(os.getenv("PROJECT_ROOT"))
project_root = Path(os.getenv("PROJECT_ROOT"))

In [3]:
# load data filepaths from config.yaml

import yaml
import os

with open(os.path.join(os.getenv("PROJECT_ROOT"), 'config.yaml'), 'r') as file:
    config = yaml.safe_load(file)

for dataset_name in config['dataset_names']:
        for key, value in config['data'][dataset_name].items():
            config['data'][dataset_name][key] = value.replace('${PROJECT_ROOT}', os.getenv("PROJECT_ROOT"))# replace the ${PROJECT_ROOT} with the actual project root


In [4]:
from src.load_data import load_data

def load_all_data(config):
    """
    Load all datasets specified in the config file.
    """
    data = dict()

    for dataset_name in config['dataset_names']:
        data[dataset_name] = dict()
        for split in ["train", "dev", "test"]:
            is_varierrnli = (dataset_name == "VariErrNLI")
            is_test = (split == "test")
            file_path = config['data'][dataset_name][f"{split}_file"]

            soft_labels, perspectivism, annotators_per_entry, ids, full_data = load_data(file_path, is_varierrnli=is_varierrnli, is_test=is_test)

            data[dataset_name][split] = {
                "soft_labels": soft_labels,
                "perspectivism": perspectivism,
                "annotators_per_entry": annotators_per_entry,
                "ids": ids,
                "data": full_data
            }

            print(f"Loaded {dataset_name} {split} data with {len(full_data)} examples.")
    
    return data

data_all_datasets = load_all_data(config)

Loaded CSC train data with 5628 examples.
Loaded CSC dev data with 704 examples.
Loaded CSC test data with 704 examples.
Loaded MP train data with 12017 examples.
Loaded MP dev data with 3005 examples.
Loaded MP test data with 3756 examples.
Loaded Paraphrase train data with 400 examples.
Loaded Paraphrase dev data with 50 examples.
Loaded Paraphrase test data with 50 examples.
Loaded VariErrNLI train data with 388 examples.
Loaded VariErrNLI dev data with 50 examples.
Loaded VariErrNLI test data with 50 examples.


In [20]:
# load the prompt template

import json

def load_prompt_template(file_path):
    """
    Load the prompt template from the config file.
    """
    data = json.load(open(file_path, 'r'))

    print(f"Prompt template version: {data['version']}")

    for dataset_name in config['dataset_names']:
        general = data["content"]
        dataset_specific = data["datasets"][dataset_name]
        introduction = general["introduction"].replace("${TASK_NAME}", dataset_specific["task_name"])
        task_description = general["task_description"].replace("${INPUT_FORMAT}", dataset_specific["input_format"]).replace("${RESPONSE_FORMAT}", dataset_specific["response_format"]).replace("${LABEL_EXPLANATION}", dataset_specific["label_explanation"])


        data["datasets"][dataset_name]["prompt_template"] = introduction + " " + task_description + " " + general["style"] + "\n" + general["examples"] + "\n" + general["input"]

        print(f"Propmt for task {dataset_name} is like:\n {data['datasets'][dataset_name]['prompt_template']}")

    return data


prompts = load_prompt_template(os.path.join(os.getenv("PROJECT_ROOT"), 'prompts/prompt_template.json'))

Prompt template version: 1.3
Propmt for task CSC is like:
 You are an expert in guessing my response against a sarcasm detection task. Your task is to analyze and predict my response to a pair of context and response between <<< and >>>, and label it with an integer from 1 to 6 where 1 means not sarcastic at all and 6 means completely sarcastic. You should reply with only the label without any additional text.
Below are some of my previous responses. You shoulad learn my reponse behaviors from them and then make the prediction. 
${EXAMPLES}
<<<
${INPUT}
>>>
Propmt for task MP is like:
 You are an expert in guessing my response against a irony detection task. Your task is to analyze and predict my response to a pair of post and reply between <<< and >>>, and label it with 0 or 1 where 0 means not ironic and 1 means ironic. You should reply with only the label without any additional text.
Below are some of my previous responses. You shoulad learn my reponse behaviors from them and then m

In [6]:
def example_prompt_generation(train_data, example_ids, annotator_id):
    """
    Select n_shots examples randomly from the train data for a specific annotator.
    """
    prompt_examples = []
    for idx, example_id in enumerate(example_ids):
        example = train_data["data"][example_id]
        label = example["annotations"][annotator_id]
        text =  "\n".join([f"[{k}]: {v}" for k, v in example["text"].items()])
        response = f"[label]: {label}"
        prompt_examples.append(f"Example {idx}:\n{text}\n{response}\n")

    return "\n".join(prompt_examples)


In [9]:
import openai
import datetime
import json
from tqdm import tqdm
import random
import os


def icl_predict(dataset_name, test_mode, train_data, test_data, prompt, model, client, n_shots, selection_method, n_entry):
    """
    intput:
        - dataset_name: name of the dataset
        - test_mode: "dev" or "test"
        - train_data: training data for the dataset
        - test_data: dev/test data for the dataset
        - prompt: prompt templates of all the datasets
        - model: model to use for prediction
        - n_shots: number of shots to use for in-context learning
        - selection_method: method to select examples for in-context learning
        - n_entry: number of test entries
    output:
        - predictions: predictions for the test data
        - logs: logs of the process
    """

    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

    logs = {
        "datetime": timestamp,
        "dataset_name": dataset_name,
        "test_mode": test_mode,
        "model": model,
        "n_shots": n_shots,
        "selection_method": selection_method,
        "predictions": dict(),
        "prompt_version": prompt["version"],
        "prompt_template": prompt["datasets"][dataset_name]["prompt_template"],
        "prompts": dict(),
        "examples_ids": dict()
    }

    predictions_all = dict()

    if n_entry != -1 and n_entry != len(test_data["ids"]):
        test_data_ids = test_data["ids"][:n_entry]
    else:
        test_data_ids = test_data["ids"]

    for test_id in tqdm(test_data_ids):
        annotators = test_data["data"][test_id]["annotators"].split(",")
        predictions = list()
        for annotator in annotators:
            if selection_method == "random":
                example_ids = [train_data["ids"][i] for i, annotator_ids in enumerate(train_data["annotators_per_entry"]) if annotator in annotator_ids]
                example_ids = random.sample(example_ids, min(n_shots, len(example_ids)))
            elif selection_method == "topk":
                example_ids = json.load(open(os.path.join(os.getenv("PROJECT_ROOT"), f"examples/{dataset_name}_{test_mode}_cosmrr_{15}.json"), "r"))[f"{test_id}+{annotator}"][:n_shots]
            else:
                raise ValueError(f"Unknown selection method: {selection_method}")
            if not example_ids:
                raise ValueError(f"No examples found for annotator {annotator} in the training data.")

            prompt_examples = example_prompt_generation(train_data, example_ids, annotator)
            prompt_template = prompt["datasets"][dataset_name]["prompt_template"]
            prompt_template = prompt_template.replace("${EXAMPLES}", prompt_examples)

            input_text = "\n".join([f"[{k}]: {v}" for k, v in test_data["data"][test_id]["text"].items()]) + "\n"
            input_text += "[label]: \n"

            prompt_template = prompt_template.replace("${INPUT}", input_text)

            logs["examples_ids"][f"{test_id}+{annotator}"] = example_ids
            logs["prompts"][f"{test_id}+{annotator}"] = prompt_template

            response = client.chat.completions.create(
                model = model,
                messages = [{
                    "role": "user",
                    "content": prompt_template
                }],
                temperature = 0.0
            ).choices[0].message.content
            response = response.replace("[label]:", "").strip()
            if dataset_name != "VariErrNLI":
                try:
                    response = int(response)
                except ValueError:
                    print(f"Warning: Response '{response}' is not an integer for test_id {test_id} and annotator {annotator}.")

            predictions.append(response)

        predictions_all[test_id] = predictions

    logs["predictions"] = predictions_all
    try:
        json.dump(logs, open(os.path.join(os.getenv("PROJECT_ROOT"), f"logs/log_icl_{dataset_name}_{test_mode}_{model}_{n_shots}_{selection_method}_{timestamp}.json"), "w"), indent=4)

        json.dump(predictions_all, open(os.path.join(os.getenv("PROJECT_ROOT"), f"predictions/pred_icl_{dataset_name}_{test_mode}_{model}_{n_shots}_{selection_method}_{timestamp}.json"), "w"), indent=4)
    except Exception as e:
        print(f"Error saving logs or predictions: {e}")
    return predictions_all, logs

In [22]:
# demo test on all datasets excep for MP

# model = "gpt-4o-mini"
model = "gpt-4o"
api_key = os.environ.get("OPENAI_API_KEY")
base_url = "https://api.openai.com/v1"

# model = "deepseek-chat"
# api_key = os.environ.get("DEEPSEEK_API_KEY")
# base_url = "https://api.deepseek.com/v1"


client = openai.OpenAI(
    api_key=api_key,
    base_url=base_url,
)

n_shots = 10
# selection_method = "random"
selection_method = "topk"
test_mode = "test"
n_entry = -1

for dataset_name in config["dataset_names"]:
    if dataset_name not in ["VariErrNLI"]:
        continue
    print(f"Running ICL for dataset {dataset_name}...")
    predictions_all, logs = icl_predict(dataset_name, test_mode, data_all_datasets[dataset_name]["train"], data_all_datasets[dataset_name][test_mode], prompts, model, client, n_shots, selection_method, n_entry)

Running ICL for dataset VariErrNLI...


 88%|████████▊ | 44/50 [03:04<00:25,  4.19s/it]


RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4o in organization org-WnNrcMUwqfyvrMWn0Op7ORNj on tokens per min (TPM): Limit 30000, Used 30000, Requested 515. Please try again in 1.03s. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'tokens', 'param': None, 'code': 'rate_limit_exceeded'}}

In [16]:
import json

def varierrnli_predictions_to_soft_labels_and_pe(predictions):
    soft_labels = dict()
    pe = dict()

    labels = ["contradiction", "entailment", "neutral"]

    for id, annotations in predictions.items():
        num_annotators = len(annotations)
        soft_label = {label: dict() for label in labels}
        for label in labels:
            count = sum(1
                        for ann in annotations
                        if label in [a.strip() for a in ann.split(',')])
            p1 = count / num_annotators
            p0 = 1 - p1
            soft_label[label] = {"0": p0, "1": p1}

        soft_labels[id] = soft_label
        
        # Initialize label vectors for each annotator
        label_vectors = {label: [0] * num_annotators for label in labels}

        # Fill in the label vectors
        for i, annotation_str in enumerate(annotations):
            for label in annotation_str.split(','):
                label = label.strip()
                if label in label_vectors:
                    label_vectors[label][i] = 1

        pe[id] = list(label_vectors.values())

    return soft_labels, pe

In [13]:
def pe_to_soft_labels(dataset_name, predictions_pe):
    """
    Convert pe predictions to soft labels.
    """
    results = dict()

    if dataset_name == "CSC":
        label_range = list(range(0, 7))
    elif dataset_name == "MP":
        label_range = [0, 1]
    else:
        label_range = list(range(-5, 6))
    count = {k: 0 for k in label_range}
    for k, v in predictions_pe.items():
        for label in v:
            count[label] += 1
        total = sum(count.values())
        soft_labels = [count[k] / total for k in label_range]
        results[k] = soft_labels

    return results

In [14]:
from src.evaluation import soft_label_evaluation, perspectivist_evaluation

def evaluate_one_dataset(dataset_name, test_mode, full_data, predictions_soft_labels, predictions_pe):
    """
    Evaluate the predictions for a specific dataset and a specific list of entries.
    """

    selected_ids = list(predictions_pe.keys())

    selected_positions = [idx for idx, _ in enumerate(full_data[dataset_name][test_mode]["ids"]) if _ in selected_ids]
    
    target_soft_labels = full_data[dataset_name][test_mode]["soft_labels"]
    target_pe = full_data[dataset_name][test_mode]["perspectivism"]

    selected_target_soft_labels = [item for idx, item in enumerate(target_soft_labels) if idx in selected_positions]
    selected_target_pe = [item for idx, item in enumerate(target_pe) if idx in selected_positions]

    if dataset_name != "VariErrNLI":
        soft_label_evaluation_results = soft_label_evaluation(dataset_name, selected_target_soft_labels, list(predictions_soft_labels.values()))
        perspectivist_evaluation_results = perspectivist_evaluation(dataset_name, selected_target_pe, list(predictions_pe.values()))
    else:
        soft_label_evaluation_results = soft_label_evaluation(dataset_name, selected_target_soft_labels, [
           [
               list(q.values()) for _, q in v.items()
           ] for _, v in predictions_soft_labels.items()
        ])
        perspectivist_evaluation_results = perspectivist_evaluation(dataset_name, selected_target_pe, [v for _, v in predictions_pe.items()])

    return {
        "soft_label_evaluation": soft_label_evaluation_results,
        "perspectivist_evaluation": perspectivist_evaluation_results
    }
    

In [35]:
predictions_pe = json.load(open(os.path.join(os.getenv("PROJECT_ROOT"), f"predictions/icl_Paraphrase_dev_gpt-4o-mini_20_random_20250701_174650.json"), "r"))
predictions_soft_labels = pe_to_soft_labels("Paraphrase", predictions_pe)


results = evaluate_one_dataset("Paraphrase", "dev", data_all_datasets, predictions_soft_labels, predictions_pe)
results

{'soft_label_evaluation': 2.83587,
 'perspectivist_evaluation': 0.18545454545454543}

In [42]:
predictions_varierrnli = json.load(open(os.path.join(os.getenv("PROJECT_ROOT"), f"predictions/icl_VariErrNLI_dev_gpt-4o-mini_10_random_20250701_153208.json"), "r"))

predictions_soft_labels_varierrnli, predictions_pe_varierrnli = varierrnli_predictions_to_soft_labels_and_pe(predictions_varierrnli)
print(predictions_soft_labels_varierrnli["59208"])
print(predictions_pe_varierrnli["59208"])

results_varierrnli = evaluate_one_dataset("VariErrNLI", "dev", data_all_datasets, predictions_soft_labels_varierrnli, predictions_pe_varierrnli)
results_varierrnli

{'contradiction': {'0': 0.75, '1': 0.25}, 'entailment': {'0': 1.0, '1': 0.0}, 'neutral': {'0': 0.25, '1': 0.75}}
[[0, 0, 1, 0], [0, 0, 0, 0], [1, 1, 0, 1]]


{'soft_label_evaluation': 0.58223,
 'perspectivist_evaluation': 0.30222219999999994}

In [79]:
import json
import datetime

def evaluate_all_datasets(preds_fp, test_mode, full_data):
    """
    Evaluate the predictions for all datasets.
    """
    results = {
        "predictions_fp": preds_fp,
        "test_mode": test_mode,
        "datasets": dict()
    }

    for dataset_name in preds_fp:
        if dataset_name != "VariErrNLI":
            predictions_pe = json.load(open(os.path.join(os.getenv("PROJECT_ROOT"), preds_fp[dataset_name]), "r"))
            predictions_soft_labels = pe_to_soft_labels(dataset_name, predictions_pe)
        else:
            predictions = json.load(open(os.path.join(os.getenv("PROJECT_ROOT"), preds_fp[dataset_name]), "r"))
            predictions_soft_labels, predictions_pe = varierrnli_predictions_to_soft_labels_and_pe(predictions)

        results["datasets"][dataset_name] = evaluate_one_dataset(dataset_name, test_mode, full_data, predictions_soft_labels, predictions_pe)

    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    json.dump(results, open(os.path.join(os.getenv("PROJECT_ROOT"), f"metrics/metrics_{timestamp}.json"), "w"), indent=4)

    return results

In [80]:
pred_fp = {
    # "CSC": "predictions/pred_icl_CSC_dev_gpt-4o-mini_10_random_20250702_030318.json",
    # "MP": "predictions/pred_icl_MP_dev_gpt-4o-mini_10_random_20250702_025229.json",
    "Paraphrase": "predictions/pred_icl_Paraphrase_dev_gpt-4o-mini_10_topk_20250703_111631.json",
    "VariErrNLI": "predictions/pred_icl_VariErrNLI_dev_gpt-4o-mini_10_topk_20250703_111831.json"
}

results = evaluate_all_datasets(pred_fp, "dev", data_all_datasets)

for dataset_name, dataset_results in results["datasets"].items():
    print(f"Results for {dataset_name}:")
    print(f"Soft Label Evaluation: {dataset_results['soft_label_evaluation']}")
    print(f"Perspectivist Evaluation: {dataset_results['perspectivist_evaluation']}")
    print()

Results for Paraphrase:
Soft Label Evaluation: 2.73878
Perspectivist Evaluation: 0.18363636363636363

Results for VariErrNLI:
Soft Label Evaluation: 0.61222
Perspectivist Evaluation: 0.3127778



In [23]:
import json
import datetime
from pathlib import Path

def to_submission_format(pred_fps):
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    
    dir_path = Path(os.getenv("PROJECT_ROOT")) / "submissions" / timestamp
    if not dir_path.exists():
        dir_path.mkdir()

    for dataset_name, pred_fp in pred_fps.items():
        if dataset_name != "VariErrNLI":
            predictions_pe = json.load(open(os.path.join(os.getenv("PROJECT_ROOT"), pred_fp), "r"))
            predictions_soft_labels = pe_to_soft_labels(dataset_name, predictions_pe)

            with open(os.path.join(os.getenv("PROJECT_ROOT"), "submissions", timestamp, f"{dataset_name}_test_soft.tsv"), "w") as f:
                for id, soft_labels in predictions_soft_labels.items():
                    f.write(f"{id}\t" + str(soft_labels) + "\n")

            with open(os.path.join(os.getenv("PROJECT_ROOT"), "submissions", timestamp, f"{dataset_name}_test_pe.tsv"), "w") as f:
                for id, pe in predictions_pe.items():
                    f.write(f"{id}\t" + str(pe) + "\n")
        else:
            predictions = json.load(open(os.path.join(os.getenv("PROJECT_ROOT"), pred_fp), "r"))
            predictions_soft_labels, predictions_pe = varierrnli_predictions_to_soft_labels_and_pe(predictions)

            with open(os.path.join(os.getenv("PROJECT_ROOT"), "submissions", timestamp, f"{dataset_name}_test_soft.tsv"), "w") as f:
                for id, soft_labels in predictions_soft_labels.items():
                    f.write(f"{id}\t" + str(
                        [list(probs.values()) for _, probs in soft_labels.items()]
                    ) + "\n")

            with open(os.path.join(os.getenv("PROJECT_ROOT"), "submissions", timestamp, f"{dataset_name}_test_pe.tsv"), "w") as f:
                for id, pe in predictions_pe.items():
                    f.write(f"{id}\t" + str(pe) + "\n")


test_pred_fps = {
    # "CSC": "predictions/pred_icl_CSC_test_gpt-4o-mini_10_topk_20250703_143039.json",
    # "MP": "predictions/pred_icl_MP_dev_gpt-4o-mini_10_random_20250702_025229.json",
    "Paraphrase": "predictions/pred_icl_Paraphrase_test_gpt-4o_10_topk_20250703_174038.json",
    # "VariErrNLI": "predictions/pred_icl_VariErrNLI_test_gpt-4o-mini_10_topk_20250703_114559.json"
}

to_submission_format(test_pred_fps)